In [1]:
# imports
import torch
from stock_dataloader import create_stock_dataloader
from lstm import StockLSTM
from transformer import StockTransformer
from model_trainer import train_model, save_model, load_model


In [2]:
# Create dataloader

# Hyperparameters
SEQ_LEN = 1250          # default 1250; window for set of time-series data points
BATCH_SIZE = 128        # default 128; uses 25-30% of 12GB NVIDIA GeForce RTX 4070 GPU
STOCKS_PER_BUCKET = 13  # default 13; number of stocks per category bucket
TRAIN_PER_BUCKET = 10   # default 10; number of training stocks per category bucket

stock_csv = 'selected_stocks_data.csv'
metadata_csv = 'selected_stocks_quality.csv'
stock_dataloader = create_stock_dataloader(stock_csv, metadata_csv, seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
                                           stocks_per_bucket=STOCKS_PER_BUCKET, train_per_bucket=TRAIN_PER_BUCKET)
train_dataloader = stock_dataloader['train_loader']

Loading stock data...

Splitting top 13 stocks per category...
✅ Train: 50 tickers | Eval: 15 tickers

Creating training sequences...

✅ Data loaded: 50 train tickers, 15 eval tickers


In [3]:
# Create LSTM Model

# Hyperparameters
INPUT_SIZE = 1          # default 1; based on data
HIDDEN_SIZE = 64        # default 64; analogous to D_MODEL; increase to 128 if underfitting
NUM_LAYERS = 2          # default 2, re-evaluate if underfitting
DROP_OUT = 0.2          # default 0.2; re-evaluate if overfitting

lstm_model = StockLSTM(input_size=INPUT_SIZE,
                  hidden_size=HIDDEN_SIZE,
                  num_layers=NUM_LAYERS,
                  dropout=DROP_OUT)

In [4]:
# Create Transformer Model

# Hyperparameters
INP_DIM = 1             # default 1; based on data
D_MODEL = 64            # default 64; analogous to HIDDEN_SIZE; re-evaluate if underfitting
N_HEADS = 4             # default 4; 64/4 = 16 - standard ratio
N_LAYERS = 3            # default 3; re-evaluate if underfitting
DIM_FEEDFORWARD = 256   # default 256; 4x D_MODEL is standard
DROPOUT = 0.1           # default 0.1; re-evaluate if overfitting
OUTPUT_DIM = 1          # default 1; based on data - next-day closing price
MAX_LEN = 1500          # default 1500; should be > SEQ_LEN

transformer_model = StockTransformer(inp_dim=INP_DIM,
                         d_model=D_MODEL,
                         n_heads=N_HEADS,
                         n_layers=N_LAYERS,
                         dim_feedforward=DIM_FEEDFORWARD,
                         dropout=DROPOUT,
                         output_dim=OUTPUT_DIM,
                         max_len=MAX_LEN)

In [ ]:
# Train model

# Hyperparameters
NUM_EPOCHS = 50         # default 50; increase if underfitting
LEARNING_RATE = 0.0001  # default 0.0001; 1e-3 exploded
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model_choice = 'Transformer'   # Select 'LSTM' or 'Transformer'
model_save_name = f'Stock{model_choice}_ModelFinal'
model_save = False
model_load = True

print(f"Training {model_choice} model on device: {DEVICE}")
if model_choice == 'LSTM':
    if model_load:
        try:
            trained_model = load_model(f'models/{model_save_name}')
        except FileNotFoundError as e:
            print(e)
            trained_model = train_model(lstm_model, train_dataloader, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, device=DEVICE)
    else:
        trained_model = train_model(lstm_model, train_dataloader, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, device=DEVICE)
    if model_save:
        save_model(trained_model, save_name=model_save_name)
elif model_choice == 'Transformer':
    if model_load:
        try:
            trained_model = load_model(f'models/{model_save_name}')
        except FileNotFoundError as e:
            print(e)
            trained_model = train_model(transformer_model, train_dataloader, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, device=DEVICE)
    else:
        trained_model = train_model(transformer_model, train_dataloader, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, device=DEVICE)
    if model_save:
        save_model(trained_model, save_name=model_save_name)
else:
    raise ValueError(f"{model_choice} is an invalid model choice. Please select 'LSTM' or 'Transformer'.")


Training Transformer model on device: cuda


Training Epochs:   0%|          | 0/50 [00:00<?, ?it/s]c:\Users\jwcabot\.conda\envs\lstm-env\Lib\site-packages\torch\nn\functional.py:5560: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
Training Epochs:   2%|▏         | 1/50 [05:21<4:22:13, 321.09s/it]

Epoch [1/50], Loss: 0.006495


Training Epochs:  20%|██        | 10/50 [52:56<3:31:12, 316.82s/it]

Epoch [10/50], Loss: 0.000539


...

Training Epochs:  80%|████████  | 40/50 [3:31:16<52:45, 316.54s/it]  

Epoch [40/50], Loss: 0.000205


Training Epochs: 100%|██████████| 50/50 [4:24:01<00:00, 316.83s/it]

Epoch [50/50], Loss: 0.000193


✅ Model saved: models\StockTransformer_ModelFinal
